# Buổi 6: Trực quan hóa dữ liệu
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Chọn loại biểu đồ phù hợp với câu hỏi kinh doanh
- Vẽ biểu đồ với Matplotlib và Seaborn; tùy chỉnh tiêu đề, nhãn, màu, chú thích
- Vẽ biểu đồ tương tác với Plotly
- Ghép nhiều biểu đồ thành một bảng điều khiển (dashboard) bán hàng

**Dữ liệu dùng trong buổi:** `supermarket_sales.csv`, `ecommerce_orders_clean.csv`, `customers.csv` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
DATA = "../data/"

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.titleweight"] = "bold"

sm = pd.read_csv(DATA + "supermarket_sales.csv", parse_dates=["ngay"])
od = pd.read_csv(DATA + "ecommerce_orders_clean.csv", parse_dates=["ngay_dat"])
od = od[od["trang_thai"] == "Hoàn thành"].copy()
od["doanh_thu"] = od["so_luong"] * od["don_gia"]

**Chọn biểu đồ theo câu hỏi:**

| Câu hỏi | Biểu đồ |
|---|---|
| Xu hướng theo thời gian? | Đường (line) |
| So sánh giữa các nhóm? | Cột / thanh ngang (bar) |
| Phân phối của một biến? | Histogram, boxplot |
| Quan hệ giữa hai biến số? | Phân tán (scatter), heatmap tương quan |
| Cơ cấu (phần của tổng thể)? | Cột chồng 100%, treemap (tròn chỉ khi ≤ 4 phần) |

## Phần A. Bài tập có hướng dẫn

### A1. Biểu đồ đường: xu hướng doanh thu theo tháng (Matplotlib)
Cấu trúc chuẩn: `fig, ax = plt.subplots()` → vẽ trên `ax` → đặt tiêu đề, nhãn trục → `plt.show()`.

In [ ]:
thang = od.set_index("ngay_dat")["doanh_thu"].resample("ME").sum() / 1e9

fig, ax = plt.subplots()
ax.plot(thang.index, thang.values, marker="o", linewidth=2, color="#1f6feb")
ax.set_title("Doanh thu thương mại điện tử theo tháng, 2025")
ax.set_xlabel("Tháng")
ax.set_ylabel("Doanh thu (tỷ đồng)")
dinh = thang.idxmax()
ax.annotate(f"Đỉnh: {thang.max():.1f} tỷ", xy=(dinh, thang.max()), xytext=(0, 10),
            textcoords="offset points", ha="center")
ax.set_ylim(0, thang.max() * 1.2)
plt.show()

### A2. Biểu đồ thanh ngang: so sánh nhóm hàng
Sắp xếp giá trị trước khi vẽ để người xem đọc thứ hạng ngay. Thanh **ngang** dễ đọc khi tên nhóm dài.

In [ ]:
nh = sm.groupby("nhom_hang")["tong_tien"].sum().sort_values() / 1e3   # triệu đồng

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(nh.index, nh.values, color="#8b949e")
bars[-1].set_color("#1f6feb")                       # tô màu nhóm cao nhất để nhấn mạnh
ax.bar_label(bars, fmt="%.0f", padding=3)
ax.set_title("Doanh thu siêu thị theo nhóm hàng (triệu đồng)")
ax.set_xlabel("")
sns.despine(left=True, bottom=True)
plt.show()

### A3. Seaborn: biểu đồ thống kê chỉ với một dòng
Seaborn làm việc trực tiếp với DataFrame: chỉ cần chỉ định `data=`, `x=`, `y=`, `hue=` (tô màu theo nhóm).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=sm, x="tong_tien", bins=30, ax=axes[0], color="#1f6feb")
axes[0].set_title("Phân phối giá trị hóa đơn (nghìn đồng)")
sns.boxplot(data=sm, x="thanh_pho", y="tong_tien", hue="loai_khach", ax=axes[1])
axes[1].set_title("Giá trị hóa đơn theo thành phố và loại khách")
plt.tight_layout(); plt.show()

In [ ]:
# Cột nhóm: doanh thu theo kênh và quý
od["quy"] = "Q" + od["ngay_dat"].dt.quarter.astype(str)
kq = od.groupby(["quy", "kenh"], as_index=False)["doanh_thu"].sum()
kq["doanh_thu"] /= 1e9
ax = sns.barplot(data=kq, x="quy", y="doanh_thu", hue="kenh")
ax.set(title="Doanh thu theo kênh và quý (tỷ đồng)", xlabel="", ylabel="")
plt.show()

In [ ]:
# Heatmap tương quan giữa các biến số
so = sm[["don_gia", "so_luong", "tong_tien", "gia_von", "loi_nhuan_gop", "danh_gia"]]
sns.heatmap(so.corr(), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Ma trận tương quan"); plt.show()

**Đọc heatmap:** gần 1 = cùng tăng, gần −1 = ngược chiều, gần 0 = không có quan hệ tuyến tính. Ở đây `danh_gia` gần như không liên quan đến số tiền.

### A4. Plotly: biểu đồ tương tác
Rê chuột để xem giá trị, kéo để phóng to, bấm vào chú thích để ẩn/hiện nhóm. Phù hợp khi trình bày trực tuyến.

In [ ]:
tn = od.assign(thang=od["ngay_dat"].dt.to_period("M").astype(str)).groupby(["thang", "nhom_hang"], as_index=False)["doanh_thu"].sum()
fig = px.line(tn, x="thang", y="doanh_thu", color="nhom_hang", markers=True,
              title="Doanh thu theo tháng và nhóm hàng", labels={"doanh_thu": "Doanh thu (đ)", "thang": "Tháng"})
fig.show()

In [ ]:
fig = px.treemap(od, path=["nhom_hang", "san_pham"], values="doanh_thu", title="Cơ cấu doanh thu theo nhóm hàng và sản phẩm")
fig.show()

### A5. Dashboard bán hàng một trang
Dùng `plt.subplots` dạng lưới để ghép 4 biểu đồ trả lời 4 câu hỏi quản lý thường hỏi.
Lưu thành ảnh bằng `fig.savefig()` để đưa vào báo cáo.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle("DASHBOARD BÁN HÀNG SIÊU THỊ 2025", fontsize=16, fontweight="bold")

t = sm.set_index("ngay")["tong_tien"].resample("ME").sum() / 1e3
ax[0, 0].plot(t.index.month, t.values, marker="o"); ax[0, 0].set_title("Doanh thu theo tháng (triệu đồng)")
ax[0, 0].set_xticks(range(1, 13))

cn = sm.groupby("thanh_pho")["tong_tien"].sum().sort_values() / 1e3
ax[0, 1].barh(cn.index, cn.values, color="#1f6feb"); ax[0, 1].set_title("Doanh thu theo thành phố (triệu đồng)")

tt = sm["thanh_toan"].value_counts(normalize=True).sort_values() * 100
ax[1, 0].barh(tt.index, tt.values, color="#2da44e"); ax[1, 0].set_title("Phương thức thanh toán (% hóa đơn)")

sns.scatterplot(data=sm, x="tong_tien", y="loi_nhuan_gop", hue="thanh_pho", alpha=0.6, ax=ax[1, 1], s=20)
ax[1, 1].set_title("Tổng tiền và lợi nhuận gộp (nghìn đồng)")

plt.tight_layout()
fig.savefig("dashboard_sieu_thi.png", dpi=150)
plt.show()

## Phần B. Bài tập tự làm

Mỗi biểu đồ phải có tiêu đề, nhãn trục có đơn vị, và **một câu nhận xét** viết trong ô Markdown bên dưới.

### Bài B1. Doanh thu theo ngày trong tuần

Dùng `sm`: vẽ biểu đồ cột doanh thu trung bình mỗi ngày theo thứ (Thứ 2 → Chủ nhật, nhãn tiếng Việt).
Tô màu khác cho ngày có doanh thu cao nhất.

In [ ]:
# Viết code của bạn ở đây


### Bài B2. So sánh phân phối điểm đánh giá

Vẽ boxplot hoặc violin plot điểm `danh_gia` theo `nhom_hang`. Nhóm hàng nào có điểm đánh giá trung vị cao nhất,
nhóm nào phân tán nhiều nhất?

In [ ]:
# Viết code của bạn ở đây


### Bài B3. Cơ cấu doanh thu theo kênh

Dùng `od`: vẽ biểu đồ cột chồng 100% thể hiện tỷ trọng doanh thu của từng kênh trong mỗi quý.
Kênh nào tăng tỷ trọng qua các quý?

<details><summary>💡 Gợi ý</summary>

Tạo pivot table quý × kênh, chia mỗi dòng cho tổng dòng (`div(..., axis=0)`), rồi `.plot(kind='bar', stacked=True)`.

</details>

In [ ]:
# Viết code của bạn ở đây


### Bài B4. Biểu đồ tương tác theo vùng

Ghép `od` với `customers.csv` để có cột `vung`. Dùng Plotly vẽ biểu đồ cột doanh thu theo nhóm hàng, tô màu theo vùng
(`barmode="group"`), có tooltip hiển thị số đơn.

In [ ]:
# Viết code của bạn ở đây


### Bài B5. Dashboard của riêng bạn

Thiết kế một dashboard 2×2 từ dữ liệu `od` trả lời 4 câu hỏi do bạn tự đặt (ví dụ: tháng nào bán tốt nhất,
nhóm hàng nào đóng góp nhiều nhất, kênh nào có giá trị đơn trung bình cao nhất, khách hàng mua bao nhiêu đơn).
Lưu thành `dashboard_tmdt.png`.

In [ ]:
# Viết code của bạn ở đây


## Phần C. Bài tập làm thêm

### Bài C1. Biểu đồ Pareto

Vẽ biểu đồ Pareto cho doanh thu theo sản phẩm (`od`): cột doanh thu giảm dần + đường tỷ trọng cộng dồn trên trục phụ
(`ax.twinx()`), kẻ đường ngang 80%. Có bao nhiêu sản phẩm tạo ra 80% doanh thu?

In [ ]:
# Viết code của bạn ở đây


### Bài C2. Heatmap tháng × ngày trong tuần

Vẽ heatmap số đơn hàng (`od`) với dòng là tháng (1–12), cột là thứ trong tuần. Có mẫu hình mùa vụ nào không?

In [ ]:
# Viết code của bạn ở đây


### Bài C3. Dashboard tương tác bằng Plotly

Dùng `plotly.subplots.make_subplots` ghép 2–4 biểu đồ Plotly vào một hình, rồi lưu thành tệp HTML
(`fig.write_html("dashboard.html")`) để gửi cho quản lý mở bằng trình duyệt.

In [ ]:
# Viết code của bạn ở đây
